# Hunyuan3D 2.1 on Kaggle, using both T4 GPUs

Redesigns each supplied reference image into a new fantasy-game asset concept, then turns the redesigned image into a textured 3D model (`.glb`) with [Tencent Hunyuan3D 2.1](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1).

**How the two GPUs are used:** the model has two stages, so each stage gets its own T4.

| GPU | Worker | Does |
|---|---|---|
| `cuda:0` | `shape_worker.py` | removes background, then image to untextured mesh (~3.3B-param model) |
| `cuda:1` | `paint_worker.py` | untextured mesh to PBR texture (multiview diffusion + upscaler + baking) |

The two run as separate processes linked by a small folder queue. While GPU 1 is texturing model *N*, GPU 0 is already
building model *N+1*. Neither stage has to share 16 GB with the other, and with a batch of images, throughput roughly doubles.
With only one GPU, the notebook runs the stages one after the other on that GPU.

### Before you run
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On** (needs a phone-verified Kaggle account)
3. Optional: **Add Input → Upload** a folder of images (PNG with a transparent background works best; other images get their background removed automatically).
4. Recommended: **Add Input → Your Datasets →** your Hunyuan3D weights dataset, so the 17.8 GB of weights don't download every session. Build it once with [`hunyuan3d_2_1_weights_builder.ipynb`](https://github.com/ZzwahwahzZ/Clash-of-Legends/blob/main/tools/ai-asset-generation/hunyuan3d_2_1_weights_builder.ipynb).
5. The new image-redesign stage uses SDXL image-to-image on GPU 1 before the Hunyuan3D workers start. Original references are preserved separately; redesigned images become the 3D inputs.\n6. **Run All.** Setup takes about 10 minutes (package installs and compiling the CUDA rasterizer), plus 5–10 minutes more if it has to download the weights.

Results go to `/kaggle/working/hy3d_out/<image name>/textured_mesh.glb` and get zipped into `/kaggle/working/hy3d_results.zip`.
For long batches, use **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab.

> License: the Tencent Hunyuan 3D 2.1 Community License applies. Read it before commercial use. It doesn't cover the EU, UK or South Korea.

## License and distribution gate — read before running

This notebook uses Tencent Hunyuan3D 2.1 under the **Tencent Hunyuan 3D 2.1 Community License Agreement**. The license excludes the EU, UK and South Korea. It restricts use and distribution of the model and its outputs outside the permitted territory, prohibits using outputs to improve another AI model, and requires license/notice disclosures when distributing products or services using the works. If the product exceeds 1 million monthly active users under the license's stated test, separate written permission is required. Read the full current license: https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1/blob/main/LICENSE

The notebook below is configured for an Egypt-based account; do not use or distribute the model or its outputs in excluded territories. The game team must include the current license and required attribution/notice with any distribution that uses the model. This is a technical reminder, not legal advice.

The notebook does not upload model weights or your Hugging Face token. It uploads only validated generated `.glb` outputs to your Hugging Face Dataset.


In [ ]:
# READ-ONLY RECOVERY INSPECTION — safe to run before any generation cell.
# This cell does not create, overwrite, move, or delete any production file.
from pathlib import Path
import shutil, json

roots = {
    "3D outputs": Path("/kaggle/working/hy3d_out"),
    "AI redesign outputs": Path("/kaggle/working/hy3d_redesign"),
    "persistent worker logs": Path("/kaggle/working/hy3d_logs"),
}
print("=== EXISTING OUTPUT LOCATIONS (READ ONLY) ===")
for label, root in roots.items():
    print(f"{label}: exists={root.exists()} | path={root}")
    if root.exists() and label == "3D outputs":
        white = sorted(root.glob("*/white_mesh.glb"))
        white_obj = sorted(root.glob("*/white_mesh.obj"))
        textured = sorted(root.glob("*/textured_mesh.glb"))
        textured_obj = sorted(root.glob("*/textured_mesh.obj"))
        print(f"  white GLB: {len(white)} | white OBJ: {len(white_obj)}")
        print(f"  textured GLB: {len(textured)} | textured OBJ: {len(textured_obj)}")
        for p in (white + textured)[:12]:
            print(f"  {p} | {p.stat().st_size / (1024**2):.2f} MB")
    elif root.exists() and label == "AI redesign outputs":
        print(f"  files: {sum(1 for p in root.rglob('*') if p.is_file())}")
    elif root.exists() and label == "persistent worker logs":
        logs = sorted(root.glob("*.log"))
        print(f"  log files: {len(logs)} | latest: {[p.name for p in logs[-5:]]}")

print("\n=== CURRENT SESSION RESOURCES ===")
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f"RAM total: {vm.total/(1024**3):.2f} GB | available: {vm.available/(1024**3):.2f} GB")
except Exception as e:
    print("RAM measurement unavailable:", repr(e))
for path in ("/kaggle/working", "/tmp"):
    try:
        usage = shutil.disk_usage(path)
        print(f"{path}: total={usage.total/(1024**3):.2f} GB | free={usage.free/(1024**3):.2f} GB")
    except Exception as e:
        print(path, "disk measurement unavailable:", repr(e))
try:
    import subprocess
    result = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv"], capture_output=True, text=True)
    print("GPU status:\n" + (result.stdout.strip() or result.stderr.strip()))
except Exception as e:
    print("GPU measurement unavailable:", repr(e))

print("\nInspection complete. No files were modified.")


In [ ]:
# Preflight: dual T4 plus either a complete attached weights Dataset or enough disk for a fresh download.
import os, glob, shutil, subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No working NVIDIA GPU detected. Select GPU T4 x2 and restart the session.')
gpus = [line.strip() for line in gpu.stdout.splitlines() if line.strip()]
print('Detected GPUs:', gpus)
if len(gpus) < 2:
    raise RuntimeError('This notebook requires two GPUs. Select GPU T4 x2 before continuing.')
if not all('T4' in line for line in gpus[:2]):
    print('WARNING: GPU type is not T4; memory and CUDA compatibility may differ.')
required = [
 'tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/model.fp16.ckpt',
 'tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1/unet/diffusion_pytorch_model.bin',
 'facebook/dinov2-giant/model.safetensors',
 'RealESRGAN_x4plus.pth', 'u2net/u2net.onnx']
attached = glob.glob('/kaggle/input/**/tencent/Hunyuan3D-2.1', recursive=True)
valid = []
for p in attached:
    root = p.split('/tencent/')[0]
    if all(os.path.isfile(os.path.join(root, rel)) for rel in required): valid.append(root)
if valid:
    print('Complete attached weights Dataset found:', valid[0])
    print('Weights will be read from the attached Dataset; the 25 GB download-space requirement is not applicable.')
else:
    free_gb = shutil.disk_usage('/tmp').free / 2**30
    print(f'No complete weights Dataset attached. Free /tmp disk: {free_gb:.1f} GB')
    if free_gb < 25:
        raise RuntimeError('No complete weights Dataset is attached and /tmp has less than 25 GB free. Build and attach the weights Dataset first; no model weights have been downloaded.')
print('Preflight passed. Continue only after reviewing the model license below.')


## 1. Check the GPUs and locate the attached weights Dataset
If a complete weights Dataset is attached, the notebook reads from `/kaggle/input` directly when there is not enough `/tmp` space to stage a local copy. If no weights Dataset is attached, step 3 downloads the weights and requires sufficient free disk.

In [ ]:
%%writefile /tmp/stage_weights.py"""Copies the weights dataset to local disk with many parallel reads.Kaggle streams dataset files over the network on first read (~17 MB/s for one sequential reader), so the modelsloaded straight from /kaggle/input in 7+ min each. Reading bounded chunks on a small worker pool runs during the install.Usage: python stage_weights.py <dataset_dir> <local_dir>"""import os, sys, threading, timefrom concurrent.futures import ThreadPoolExecutorsrc, dst = sys.argv[1], sys.argv[2]CHUNK = 16 << 20  # bounded in-flight workfiles = [os.path.join(d, f) for d, _, fs in os.walk(src) for f in fs]total = sum(os.path.getsize(p) for p in files)jobs = []for p in files:    out = os.path.join(dst, os.path.relpath(p, src))    os.makedirs(os.path.dirname(out), exist_ok=True)    size = os.path.getsize(p)    with open(out, "wb") as f:        f.truncate(size)    jobs += [(p, out, off, min(CHUNK, size - off)) for off in range(0, size, CHUNK)]jobs.sort(key=lambda j: j[2])  # spread early chunks across all filescopied, lock = 0, threading.Lock()def copy(job):    global copied    p, out, off, n = job    with open(p, "rb") as fi, open(out, "r+b") as fo:        fi.seek(off)        fo.seek(off)        while n:            buf = fi.read(min(n, 4 << 20))            if not buf:                raise IOError(f"short read: {p}")            fo.write(buf)            n -= len(buf)            with lock:                copied += len(buf)def report():    while copied < total:        time.sleep(30)        print(f"staged {copied / 2**30:.1f} / {total / 2**30:.1f} GB "              f"({copied / 2**20 / (time.time() - start):.0f} MB/s)", flush=True)start = time.time()threading.Thread(target=report, daemon=True).start()with ThreadPoolExecutor(max_workers=3) as pool:    list(pool.map(copy, jobs))open(os.path.join(dst, "_STAGED"), "w").close()print(f"STAGED {total / 2**30:.1f} GB in {time.time() - start:.0f}s", flush=True)

In [ ]:
import glob, os, shutil, subprocess, sys
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

WEIGHTS = "/tmp/hy3d-weights"
found = glob.glob("/kaggle/input/**/tencent/Hunyuan3D-2.1", recursive=True)
DATASET = found[0].split("/tencent/")[0] if found else None
stage_proc = None
if DATASET and not os.path.exists(f"{WEIGHTS}/_STAGED"):
    size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(DATASET) for f in fs)
    free = shutil.disk_usage("/tmp").free
    if free > size + (5 << 30):
        print(f"Copying {size / 2**30:.1f} GB from {DATASET} to {WEIGHTS} in the background...")
        stage_proc = subprocess.Popen([sys.executable, "/tmp/stage_weights.py", DATASET, WEIGHTS],
                                      stdout=open("/tmp/stage_weights.log", "w"), stderr=subprocess.STDOUT)
    else:
        print(f"Only {free / 2**30:.0f} GB free in /tmp - models will load from the dataset directly (slower).")
        WEIGHTS = DATASET
elif not DATASET:
    print("No weights dataset attached - weights will be downloaded in step 3.")

## 2. Install (about 10 minutes)
Clones the repo to `/tmp`, installs pinned dependencies on top of Kaggle's PyTorch, and compiles the CUDA rasterizer and mesh inpainter for the GPUs in this session.

In [ ]:
!mkdir -p /tmp/hy3d

In [ ]:
%%writefile /tmp/hy3d/install.sh
#!/usr/bin/env bash
# Installs Hunyuan3D 2.1 on top of Kaggle's preinstalled PyTorch (no torch reinstall).
set -euo pipefail
PY=${PY:-python}
REPO=${REPO:-/tmp/Hunyuan3D-2.1}
BUILD_CUDA=${BUILD_CUDA:-1}
mkdir -p /tmp/hy3d

[ -d "$REPO" ] || git clone -q --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git "$REPO"
cd "$REPO"
"$PY" -c "import sys, torch; print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda)"

# numpy 1.26 has no wheels for Python 3.13+, where numpy 2.2 is tested to work.
"$PY" -c "import sys; print('numpy==1.26.4' if sys.version_info < (3, 13) else 'numpy<2.3')" > /tmp/hy3d/constraints.txt
PIP="$PY -m pip install -q --root-user-action=ignore -c /tmp/hy3d/constraints.txt"

echo "Installing Python packages..."
$PIP numpy \
  diffusers==0.30.0 transformers==4.46.0 accelerate==1.1.1 huggingface-hub==0.30.2 hf_transfer \
  pytorch-lightning==1.9.5 safetensors einops==0.8.0 omegaconf==2.3.0 pyyaml tqdm psutil configargparse \
  opencv-python==4.10.0.84 opencv-python-headless==4.10.0.84 scikit-image imageio \
  trimesh==4.4.7 "pymeshlab>=2022.2.post3" pygltflib==1.16.3 xatlas \
  rembg==2.0.65 onnxruntime timm torchdiffeq ninja pybind11 spandrel

# bpy (Blender) has no wheel for this Python and is only used by an OBJ->GLB helper we replace.
sed -i 's/^import bpy$/try:\n    import bpy\nexcept ImportError:\n    bpy = None/' hy3dpaint/DifferentiableRenderer/mesh_utils.py
# basicsr/realesrgan (the texture upscaler) fail to install on Python 3.13; swap in spandrel.
cat > hy3dpaint/utils/image_super_utils.py <<'PYEOF'
# Replaces the basicsr/realesrgan upscaler: neither package installs on Python 3.13+.
# spandrel loads the same RealESRGAN_x4plus.pth weights.
import numpy as np
import torch
from PIL import Image
from spandrel import ImageModelDescriptor, ModelLoader


class imageSuperNet:
    def __init__(self, config) -> None:
        model = ModelLoader().load_from_file(config.realesrgan_ckpt_path)
        assert isinstance(model, ImageModelDescriptor)
        device = torch.device(config.device)
        dtype = torch.float16 if device.type == "cuda" else torch.float32
        self.model = model.to(device).to(dtype).eval()
        self.device, self.dtype = device, dtype

    @torch.no_grad()
    def __call__(self, image):
        x = torch.from_numpy(np.array(image.convert("RGB"))).permute(2, 0, 1)[None]
        y = self.model(x.to(self.device, self.dtype) / 255.0).clamp(0, 1)
        return Image.fromarray((y[0].permute(1, 2, 0).float().cpu().numpy() * 255).round().astype(np.uint8))
PYEOF

EXT=$("$PY" -c "import sysconfig; print(sysconfig.get_config_var('EXT_SUFFIX'))")
echo "Building mesh inpainter..."
(cd hy3dpaint/DifferentiableRenderer && \
  c++ -O3 -shared -std=c++11 -fPIC $("$PY" -m pybind11 --includes) mesh_inpaint_processor.cpp -o "mesh_inpaint_processor$EXT")

if [ "$BUILD_CUDA" = 1 ]; then
  export CUDA_HOME=${CUDA_HOME:-/usr/local/cuda}
  export PATH="$CUDA_HOME/bin:$PATH"
  command -v nvcc >/dev/null || { echo "nvcc not found in $CUDA_HOME/bin - is a GPU accelerator enabled?"; exit 1; }
  # Compile only for the GPUs in this session (T4 = 7.5, P100 = 6.0): much faster than all archs.
  export TORCH_CUDA_ARCH_LIST=$("$PY" -c "import torch; print(';'.join(sorted({'%d.%d' % torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())})))")
  echo "Building custom rasterizer for CUDA arch $TORCH_CUDA_ARCH_LIST (takes a few minutes)..."
  (cd hy3dpaint/custom_rasterizer && $PY -m pip install -q --root-user-action=ignore --no-build-isolation .)
fi

echo "Checking imports..."
"$PY" - <<'EOF'
import os, sys
sys.path[:0] = ["hy3dshape", "hy3dpaint"]
from DifferentiableRenderer import mesh_inpaint_processor
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
if os.environ.get("BUILD_CUDA", "1") == "1":
    import custom_rasterizer_kernel
    from textureGenPipeline import Hunyuan3DPaintPipeline
from convert_utils import create_glb_with_pbr_materials
import numpy
print("Install OK (numpy %s)" % numpy.__version__)
EOF

In [ ]:
import os, subprocess, sys
# subprocess (not !bash) so a failed install stops Run All instead of scrolling past.
subprocess.run(["bash", "/tmp/hy3d/install.sh"], env=dict(os.environ, PY=sys.executable), check=True)

## 3. Weights: attached dataset, or download (17.8 GB)
If you attached the weights dataset (built once with `hunyuan3d_2_1_weights_builder.ipynb`), this waits for the
background copy started in step 1 and downloads nothing. Without one, it downloads everything to `/tmp`,
which takes about 5–10 minutes per session.

In [ ]:
%%writefile /tmp/hy3d/prefetch.py"""Downloads every weight file (~17.8 GB) into one folder, in parallel.Usage: python prefetch.py <weights_dir>Layout (the workers and the Kaggle weights dataset both use it):  <weights_dir>/tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/       shape model      6.9 GB  <weights_dir>/tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1/  texture model    6.4 GB  <weights_dir>/facebook/dinov2-giant/                          texture encoder  4.2 GB  <weights_dir>/RealESRGAN_x4plus.pth                           upscaler  <weights_dir>/u2net/u2net.onnx                                background removal"""import os, shutil, sys, urllib.requestfrom concurrent.futures import ThreadPoolExecutortry:    import hf_transfer  # noqa: F401  (several times faster on Kaggle)    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"except ImportError:    passfrom huggingface_hub import snapshot_downloadW = sys.argv[1]DIT = "tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/model.fp16.ckpt"PAINT = "tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1/unet/diffusion_pytorch_model.bin"DINO = "facebook/dinov2-giant/model.safetensors"ESRGAN = "RealESRGAN_x4plus.pth"U2NET = "u2net/u2net.onnx"def have(rel):    return os.path.exists(os.path.join(W, rel))def hub(repo_id, patterns):    # One call per repo: parallel calls sharing a local_dir clobber each other's temp files.    if patterns:        snapshot_download(repo_id=repo_id, allow_patterns=patterns, local_dir=os.path.join(W, repo_id))    return repo_iddef url(link, rel):    dest = os.path.join(W, rel)    if not os.path.exists(dest):        os.makedirs(os.path.dirname(dest), exist_ok=True)        urllib.request.urlretrieve(link, dest + ".part")        os.replace(dest + ".part", dest)    return rel# Anything already present (e.g. linked in from a partial dataset) is skipped.tasks = [    lambda: hub("tencent/Hunyuan3D-2.1", ([] if have(DIT) else ["hunyuan3d-dit-v2-1/*"])                + ([] if have(PAINT) else ["hunyuan3d-paintpbr-v2-1/*"])),    lambda: hub("facebook/dinov2-giant", [] if have(DINO) else ["*.json", "*.safetensors", "*.txt"]),    lambda: url("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth", ESRGAN),    lambda: url("https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx", U2NET),]with ThreadPoolExecutor(max_workers=2) as pool:    for done in pool.map(lambda task: task(), tasks):        print("ready:", done)for repo_id in ("tencent/Hunyuan3D-2.1", "facebook/dinov2-giant"):  # download bookkeeping only    shutil.rmtree(os.path.join(W, repo_id, ".cache"), ignore_errors=True)missing = [rel for rel in (DIT, PAINT, DINO, ESRGAN, U2NET) if not have(rel)]if missing:    sys.exit(f"Download incomplete, missing: {missing}")size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(W) for f in fs)print(f"All weights present: {size / 2**30:.1f} GB in {W}")

In [ ]:
import os, subprocess, sys, time

REQUIRED = ["tencent/Hunyuan3D-2.1/hunyuan3d-dit-v2-1/model.fp16.ckpt",
            "tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1/unet/diffusion_pytorch_model.bin",
            "facebook/dinov2-giant/model.safetensors",
            "RealESRGAN_x4plus.pth",
            "u2net/u2net.onnx"]

def stage_status():
    lines = open("/tmp/stage_weights.log").read().strip().splitlines()
    return lines[-1] if lines else "copying weights to local disk..."

if stage_proc is not None:  # started in step 1; usually done by now
    while stage_proc.poll() is None:
        print(stage_status())
        time.sleep(30)
    print(stage_status())
    if stage_proc.returncode != 0:
        print("Copy failed - loading from the dataset directly instead.")
        WEIGHTS = DATASET

missing = [f for f in REQUIRED if not os.path.exists(f"{WEIGHTS}/{f}")]
if missing:
    if WEIGHTS == DATASET:  # never write into the read-only dataset
        WEIGHTS = "/tmp/hy3d-weights"
        subprocess.run(["cp", "-rsf", f"{DATASET}/.", WEIGHTS], check=True)  # symlinks, no copying
    print(f"Missing {missing} - downloading only those" if DATASET else
          "No weights dataset attached - downloading everything (attach one to skip this next time).")
    subprocess.run([sys.executable, "/tmp/hy3d/prefetch.py", WEIGHTS], check=True)
OFFLINE = True  # every file is present locally now
print("Weights:", WEIGHTS)

## 4. Worker scripts
Each one runs in its own process, pinned to one GPU with `CUDA_VISIBLE_DEVICES`.

In [ ]:
%%writefile /tmp/hy3d/shape_worker.py"""GPU worker 1: image -> untextured mesh. Each finished mesh is queued for paint_worker.py."""import json, os, sys, time, traceback, gcimport trimeshcfg = json.load(open(sys.argv[1]))os.chdir(cfg["repo"])sys.path.insert(0, os.path.join(cfg["repo"], "hy3dshape"))import torchfrom PIL import Imagefrom hy3dshape import FaceReducerfrom hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipelinefrom hy3dshape.rembg import BackgroundRemoverdef log(msg):    print(f"[shape {time.strftime('%H:%M:%S')}] {msg}", flush=True)def enqueue(job):    tmp = os.path.join(cfg["queue"], f".{job['name']}.tmp")    with open(tmp, "w") as f:        json.dump(job, f)    os.replace(tmp, os.path.join(cfg["queue"], f"{job['name']}.json"))  # atomic: paint never sees half a fileimport faulthandlerfaulthandler.dump_traceback_later(300, repeat=True)  # while loading: log the stack every 5 min, so a stall shows wheret = time.time()pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(    "tencent/Hunyuan3D-2.1", subfolder="hunyuan3d-dit-v2-1", use_safetensors=False, device="cuda")rembg = BackgroundRemover()face_reducer = FaceReducer()faulthandler.cancel_dump_traceback_later()log(f"model loaded on {torch.cuda.get_device_name(0)} in {time.time() - t:.0f}s")open(os.path.join(cfg["queue"], "_SHAPE_READY"), "w").close()for item in cfg["images"]:    name, out = item["name"], os.path.join(cfg["out"], item["name"])    os.makedirs(out, exist_ok=True)    try:        existing_obj = os.path.join(out, "white_mesh.obj")        existing_input = os.path.join(out, "input.png")        if cfg["skip_existing"] and os.path.isfile(existing_obj) and os.path.isfile(existing_input) and os.path.getsize(existing_obj) > 1024:            try:                cached_mesh = trimesh.load(existing_obj, force="mesh", process=False)                cache_valid = len(cached_mesh.vertices) > 0 and len(cached_mesh.faces) > 0            except Exception:                cache_valid = False            if cache_valid:                log(f"{name}: verified existing white mesh, skipping shape generation")                enqueue({"name": name, "dir": out})                continue            log(f"{name}: existing white mesh is incomplete; regenerating safely")        t = time.time()        torch.cuda.reset_peak_memory_stats()        with Image.open(item["path"]) as source_image:            image = source_image.copy()        if image.mode != "RGBA" or cfg["force_rembg"]:            processed_image = rembg(image.convert("RGB"))            image.close()            image = processed_image        image.save(os.path.join(out, "input.png"))        mesh = pipe(            image=image,            num_inference_steps=cfg["steps"],            guidance_scale=cfg["guidance_scale"],            generator=torch.Generator().manual_seed(cfg["seed"]),            octree_resolution=cfg["octree_resolution"],            num_chunks=cfg["num_chunks"],            output_type="trimesh",        )[0]        mesh = face_reducer(mesh, max_facenum=cfg["max_faces"])        # Write temporary files first; publish final names only after each export completes.        tmp_glb = os.path.join(out, ".white_mesh.tmp.glb")        tmp_obj = os.path.join(out, ".white_mesh.tmp.obj")        mesh.export(tmp_glb)        mesh.export(tmp_obj, include_normals=False)        os.replace(tmp_glb, os.path.join(out, "white_mesh.glb"))        os.replace(tmp_obj, os.path.join(out, "white_mesh.obj"))        log(f"{name}: {len(mesh.faces)} faces in {time.time() - t:.0f}s "            f"(peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.1f} GB)")        enqueue({"name": name, "dir": out})    except Exception:        log(f"{name}: FAILED\n{traceback.format_exc()}")    finally:        # This worker runs at module scope; drop large per-asset intermediates.        _image_ref = globals().pop("image", None)        if _image_ref is not None:            try:                _image_ref.close()            except Exception:                pass        for _name in ("mesh", "cached_mesh"):            globals().pop(_name, None)        gc.collect()        torch.cuda.empty_cache()        try:            torch.cuda.ipc_collect()        except Exception:            passopen(os.path.join(cfg["queue"], "_SHAPE_DONE"), "w").close()log("all shapes done")

In [ ]:
%%writefile /tmp/hy3d/paint_worker.py"""GPU worker 2: untextured mesh + reference image -> PBR-textured GLB, fed by shape_worker.py."""import glob, json, os, sys, time, traceback, struct, gccfg = json.load(open(sys.argv[1]))os.chdir(cfg["repo"])sys.path[:0] = [cfg["repo"], os.path.join(cfg["repo"], "hy3dpaint")]import base64import pygltflibimport pymeshlabimport torchimport textureGenPipelinefrom textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfigfrom convert_utils import create_glb_with_pbr_materialsdef log(msg):    print(f"[paint {time.strftime('%H:%M:%S')}] {msg}", flush=True)def remesh_to_max_faces(mesh_path, remesh_path):    """Replaces the repo's remesh_mesh, which cuts every mesh to 40k faces (with open3d, which isn't installed).    That cap silently threw away the detail a higher max_faces setting asked for; this keeps up to max_faces.    """    ms = pymeshlab.MeshSet()    ms.load_new_mesh(mesh_path)    if ms.current_mesh().face_number() > cfg["max_faces"]:        ms.meshing_decimation_quadric_edge_collapse(targetfacenum=cfg["max_faces"])    ms.save_current_mesh(remesh_path, save_textures=False)textureGenPipeline.remesh_mesh = remesh_to_max_faces  # the pipeline looks it up as a module global at call timedef embed_textures(path):    """Moves textures into the GLB binary chunk with the right mimeType.    create_glb_with_pbr_materials stores them as JSON data URIs all labelled image/png (the base colour is a JPEG).    Web viewers accept that, but it is invalid GLB: Windows 3D Viewer and stricter importers show no texture.    """    gltf = pygltflib.GLTF2().load(path)    blob = bytearray(gltf.binary_blob() or b"")    changed = False    for image in gltf.images:        if not (image.uri or "").startswith("data:"):            continue        data = base64.b64decode(image.uri.split(",", 1)[1])        blob += b"\x00" * (-len(blob) % 4)  # bufferViews must be 4-byte aligned        gltf.bufferViews.append(pygltflib.BufferView(buffer=0, byteOffset=len(blob), byteLength=len(data)))        blob += data        mime = "image/jpeg" if data[:3] == b"\xff\xd8\xff" else "image/png"        image.uri, image.bufferView, image.mimeType = None, len(gltf.bufferViews) - 1, mime        changed = True    if not changed:        return    for material in gltf.materials:        if material.alphaMode in (None, "OPAQUE"):            material.alphaCutoff = None  # only meaningful with alphaMode MASK    blob += b"\x00" * (-len(blob) % 4)    gltf.buffers[0].byteLength = len(blob)    gltf.set_binary_blob(bytes(blob))    gltf.save_binary(path)def next_job():    jobs = sorted(glob.glob(os.path.join(cfg["queue"], "*.json")), key=os.path.getmtime)    return jobs[0] if jobs else None# The texture model always calls snapshot_download(); serve it from the weights folder instead.import huggingface_hub_download = huggingface_hub.snapshot_downloaddef _local_first(repo_id, *args, **kwargs):    local = os.path.join(cfg["weights"], repo_id)    return local if os.path.isdir(local) else _download(repo_id, *args, **kwargs)huggingface_hub.snapshot_download = _local_first  # multiview_utils looks it up at call timeimport faulthandlerfaulthandler.dump_traceback_later(300, repeat=True)  # while loading: log the stack every 5 min, so a stall shows wheret = time.time()conf = Hunyuan3DPaintConfig(cfg["texture_views"], cfg["texture_resolution"])conf.dino_ckpt_path = os.path.join(cfg["weights"], "facebook/dinov2-giant")conf.realesrgan_ckpt_path = os.path.join(cfg["weights"], "RealESRGAN_x4plus.pth")conf.multiview_cfg_path = "hy3dpaint/cfgs/hunyuan-paint-pbr.yaml"conf.custom_pipeline = "hy3dpaint/hunyuanpaintpbr"pipe = Hunyuan3DPaintPipeline(conf)faulthandler.cancel_dump_traceback_later()log(f"model loaded on {torch.cuda.get_device_name(0)} in {time.time() - t:.0f}s")while True:    job_file = next_job()    if job_file is None:        # Shape writes _SHAPE_DONE after its last enqueue, so re-check the queue once it appears.        if os.path.exists(os.path.join(cfg["queue"], "_SHAPE_DONE")) and next_job() is None:            break        time.sleep(2)        continue    job = json.load(open(job_file))    os.remove(job_file)    name, out = job["name"], job["dir"]    glb_path = os.path.join(out, "textured_mesh.glb")    if cfg["skip_existing"] and os.path.isfile(glb_path):        try:            with open(glb_path, "rb") as existing_file:                header = existing_file.read(12)            existing_valid = (len(header) == 12 and header[:4] == b"glTF"                              and struct.unpack("<I", header[4:8])[0] == 2                              and struct.unpack("<I", header[8:12])[0] == os.path.getsize(glb_path))        except OSError:            existing_valid = False        if existing_valid:            embed_textures(glb_path)            log(f"{name}: verified textured GLB already exists, skipping")            gc.collect()            torch.cuda.empty_cache()            continue        log(f"{name}: incomplete textured GLB detected; regenerating to a temporary file")    try:        t = time.time()        torch.cuda.reset_peak_memory_stats()        obj_path = pipe(            mesh_path=os.path.join(out, "white_mesh.obj"),            image_path=os.path.join(out, "input.png"),            output_mesh_path=os.path.join(out, "textured_mesh.obj"),            save_glb=False,        )        base = obj_path[:-len(".obj")]        tmp_glb_path = os.path.join(out, ".textured_mesh.tmp.glb")        create_glb_with_pbr_materials(            obj_path,            {"albedo": base + ".jpg", "metallic": base + "_metallic.jpg", "roughness": base + "_roughness.jpg"},            tmp_glb_path,        )        embed_textures(tmp_glb_path)        with open(tmp_glb_path, "rb") as generated_file:            generated_header = generated_file.read(12)        if (len(generated_header) != 12 or generated_header[:4] != b"glTF"                or struct.unpack("<I", generated_header[4:8])[0] != 2                or struct.unpack("<I", generated_header[8:12])[0] != os.path.getsize(tmp_glb_path)):            raise RuntimeError("Generated GLB failed header/length validation; final output was not replaced.")        os.replace(tmp_glb_path, glb_path)        log(f"{name}: textured in {time.time() - t:.0f}s "            f"(peak VRAM {torch.cuda.max_memory_allocated() / 2**30:.1f} GB)")    except torch.cuda.OutOfMemoryError:        log(f"{name}: OUT OF GPU MEMORY - lower texture_views / texture_resolution in the settings cell")    except Exception:        log(f"{name}: FAILED\n{traceback.format_exc()}")    finally:        for _name in ("obj_path", "base", "tmp_glb_path", "generated_header"):            globals().pop(_name, None)        gc.collect()        torch.cuda.empty_cache()        try:            torch.cuda.ipc_collect()        except Exception:            passlog("all textures done")

## 5. Settings and input images

In [ ]:
import glob, os, re, shutil, zipfile, hashlibfrom pathlib import Pathfrom PIL import ImageREPO = "/tmp/Hunyuan3D-2.1"OUT_DIR = "/kaggle/working/hy3d_out"IMAGE_EXTRACT_DIR = Path("/kaggle/working/conquer_original_reference_images")IMAGE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)SETTINGS = dict(    # shape (GPU 0)    steps=30,    guidance_scale=5.0,    octree_resolution=256,    num_chunks=8000,    seed=1234,    max_faces=40000,    force_rembg=False,    # texture (GPU 1); conservative settings for a 16 GB T4.    texture_views=6,    texture_resolution=512,    skip_existing=True,)MAX_IMAGES = 50IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp")# Locate the attached Hunyuan weights Dataset so its files are never treated as reference images.weights_root = Path(DATASET).resolve() if DATASET else Nonedef is_inside(path, root):    try:        Path(path).resolve().relative_to(Path(root).resolve())        return True    except (ValueError, OSError):        return False# Extract attached reference-image ZIP archives safely. The weights Dataset is excluded.zip_files = []for z in Path("/kaggle/input").rglob("*.zip"):    if weights_root and is_inside(z, weights_root):        continue    zip_files.append(z)# Check the total uncompressed ZIP size before extracting anything.zip_uncompressed_bytes = 0for archive in zip_files:    try:        with zipfile.ZipFile(archive) as zf:            zip_uncompressed_bytes += sum(info.file_size for info in zf.infolist() if not info.is_dir())    except zipfile.BadZipFile:        passworking_free_bytes = shutil.disk_usage(IMAGE_EXTRACT_DIR).freeZIP_DISK_RESERVE_BYTES = 1 << 30if zip_uncompressed_bytes + ZIP_DISK_RESERVE_BYTES > working_free_bytes:    raise RuntimeError(        f"Safe stop before ZIP extraction: need about {zip_uncompressed_bytes / 2**30:.1f} GB "        f"plus 1 GB reserve, but only {working_free_bytes / 2**30:.1f} GB is free."    )for archive in sorted(zip_files):    try:        with zipfile.ZipFile(archive) as zf:            for member in zf.infolist():                if not member.filename or member.is_dir():                    continue                target = (IMAGE_EXTRACT_DIR / member.filename).resolve()                if not is_inside(target, IMAGE_EXTRACT_DIR):                    raise RuntimeError(f"Unsafe path in ZIP archive: {member.filename}")            zf.extractall(IMAGE_EXTRACT_DIR)        print(f"Extracted reference archive: {archive.name}")    except zipfile.BadZipFile:        print(f"Skipping invalid ZIP: {archive}")# Collect supported images from attached Kaggle inputs and the extracted reference archive.candidate_roots = [Path("/kaggle/input"), IMAGE_EXTRACT_DIR]paths = []for root in candidate_roots:    if not root.exists():        continue    for p in root.rglob("*"):        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS:            continue        if weights_root and is_inside(p, weights_root):            continue        paths.append(str(p))# Verify image contents and remove exact duplicates by SHA-256.images, seen_hashes, seen_names = [], set(), set()invalid_images = []def sha256_file_stream(path, chunk_bytes=4 << 20):    digest = hashlib.sha256()    with open(path, "rb") as stream:        for block in iter(lambda: stream.read(chunk_bytes), b""):            digest.update(block)    return digest.hexdigest()for p in sorted(set(paths)):    try:        digest = sha256_file_stream(p)        if digest in seen_hashes:            continue        with Image.open(p) as im:            im.verify()        seen_hashes.add(digest)        base = re.sub(r"[^A-Za-z0-9_-]+", "_", Path(p).stem).strip("_") or "reference"        name, suffix = base, 1        while name in seen_names:            suffix += 1            name = f"{base}_{suffix}"        seen_names.add(name)        images.append({"name": name, "path": p})    except Exception as exc:        invalid_images.append((p, str(exc)))if invalid_images:    print(f"Skipped {len(invalid_images)} unreadable or unsupported image file(s).")if not images:    raise RuntimeError(        "No valid reference images were found. In Kaggle, attach the "        "'Conquer Original Reference Images' Dataset (ZIP or extracted image files), "        "then rerun this cell. The Hunyuan3D weights Dataset is detected and excluded."    )images = images[:MAX_IMAGES]print(f"Ready: {len(images)} unique, verified reference image(s).")for item in images:    print(f"  {item['name']}: {item['path']}")

## 5A. AI redesign of reference images (before 3D generation)
This stage preserves each original input, creates a newly designed image with SDXL image-to-image, saves a side-by-side comparison and metrics, then changes each image path so the existing Hunyuan3D workers consume the redesigned image. The similarity score is a visual diagnostic, not a legal clearance test. GPU 1 is released before the Hunyuan3D workers start.

In [ ]:
# Stage 5A: create genuinely redesigned image inputs before Hunyuan3D.# The source images remain untouched. Hunyuan3D receives only the newly generated images.from pathlib import Pathfrom PIL import Image, ImageOps, ImageDrawimport json, os, re, gc, timeimport numpy as npimport torchfrom diffusers import AutoPipelineForImage2Imagefrom skimage.metrics import structural_similarityREDESIGN_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"REDESIGN_ROOT = Path("/kaggle/working/hy3d_redesign")REDESIGN_ROOT.mkdir(parents=True, exist_ok=True)REDESIGN_STRENGTH = 0.64REDESIGN_STEPS = 28REDESIGN_GUIDANCE = 6.5REDESIGN_SEED = 73184REDESIGN_MAX_SIDE = 1024REDESIGN_RETRY_SIMILARITY = 0.93REDESIGN_PROMPTS = {}  # Optional: {"asset_file_name": "your asset-specific redesign prompt"}REDESIGN_SKIP_EXISTING = Trueif torch.cuda.device_count() < 2:    raise RuntimeError("The existing dual-T4 notebook requires two GPUs; image redesign is assigned to GPU 1.")import psutil, subprocessram_before_sdxl_gb = psutil.virtual_memory().available / (1024 ** 3)gpu_before_sdxl = subprocess.run(    ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],    capture_output=True, text=True, check=True).stdout.strip().splitlines()gpu1_free_mb = int(gpu_before_sdxl[1].split(",")[1].strip())print(f"SDXL preflight: host RAM available={ram_before_sdxl_gb:.1f} GB | GPU 1 free={gpu1_free_mb/1024:.1f} GB")if ram_before_sdxl_gb < 12 or gpu1_free_mb < 7000:    raise RuntimeError("Safe stop before SDXL load: insufficient free host RAM or GPU 1 VRAM. No redesign was started.")def prepare_sdxl_image(source_path):    with Image.open(source_path) as source:        im = source.copy()    if im.mode == "RGBA":        bg = Image.new("RGB", im.size, (128, 128, 128))        bg.paste(im, mask=im.getchannel("A"))        im = bg    else:        im = im.convert("RGB")    # Keep the complete object and its aspect ratio; pad rather than crop.    im = ImageOps.pad(im, (1024, 1024), method=Image.Resampling.LANCZOS, color=(128, 128, 128), centering=(0.5, 0.5))    return imdef image_metrics(original, redesigned):    a = np.asarray(ImageOps.fit(original.convert("RGB"), (512, 512)), dtype=np.uint8)    b = np.asarray(ImageOps.fit(redesigned.convert("RGB"), (512, 512)), dtype=np.uint8)    similarity = float(structural_similarity(a, b, channel_axis=2, data_range=255))    mean_abs_diff = float(np.abs(a.astype(np.float32) - b.astype(np.float32)).mean())    return {"ssim_similarity": similarity, "mean_absolute_pixel_difference": mean_abs_diff}def asset_prompt(name):    custom = REDESIGN_PROMPTS.get(name)    if custom:        return custom    category = re.sub(r"[_-]+", " ", name).strip()    return (        f"Create a genuinely original, production-quality fantasy MMORPG game asset concept for: {category}. "        "Preserve only the broad gameplay function and subject category. Invent a new silhouette, proportions, "        "construction, surface geometry, ornaments, materials, textures, and decorative language. "        "Do not reproduce the exact reference design or its distinctive details. "        "Classic colorful East Asian fantasy game atmosphere, coherent stylized 3D game-art presentation, "        "single complete subject, centered, full object visible, three-quarter view, neutral plain studio background, "        "clean readable shape, detailed craftsmanship, no text, no logo, no watermark."    )print("Loading SDXL image-to-image model on GPU 1. This runs before either Hunyuan3D worker starts.")redesign_pipe = AutoPipelineForImage2Image.from_pretrained(    REDESIGN_MODEL,    torch_dtype=torch.float16,    variant="fp16",    use_safetensors=True,    add_watermarker=False,)redesign_pipe.enable_model_cpu_offload(gpu_id=1)redesign_pipe.enable_vae_slicing()redesign_pipe.enable_vae_tiling()redesign_pipe.enable_attention_slicing("max")print("SDXL memory safeguards enabled: CPU offload, VAE slicing/tiling, attention slicing.")redesign_reports = []for index, item in enumerate(images):    live_ram_gb = psutil.virtual_memory().available / (1024 ** 3)    live_gpu = subprocess.run(        ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],        capture_output=True, text=True    )    live_gpu_rows = live_gpu.stdout.strip().splitlines() if live_gpu.returncode == 0 else []    gpu1_live_free_gb = (int(live_gpu_rows[1].split(",")[1].strip()) / 1024                         if len(live_gpu_rows) > 1 else 0)    print(f"Before redesign item {index+1}/{len(images)}: RAM available={live_ram_gb:.1f} GB | GPU 1 free={gpu1_live_free_gb:.2f} GB")    if live_ram_gb < 9.0 or gpu1_live_free_gb < 0.8:        raise RuntimeError("Safe stop between redesign items: memory safety floor reached. Completed redesign files are preserved; rerun this stage to resume.")    name = item["name"]    original_path = Path(item["path"])    asset_dir = REDESIGN_ROOT / name    asset_dir.mkdir(parents=True, exist_ok=True)    original_copy = asset_dir / "reference_original.png"    redesigned_path = asset_dir / "redesigned_input.png"    comparison_path = asset_dir / "original_vs_redesigned.png"    if not original_path.is_file():        raise FileNotFoundError(f"Input image is missing: {original_path}")    with Image.open(original_path) as source_image:        original = source_image.convert("RGB")    original.save(original_copy)    if REDESIGN_SKIP_EXISTING and redesigned_path.is_file():        redesigned = Image.open(redesigned_path).convert("RGB")        print(f"{name}: reusing completed redesigned image.")    else:        init_image = prepare_sdxl_image(original_path)        prompt = asset_prompt(name)        result = redesign_pipe(            prompt=prompt,            negative_prompt="exact copy, identical design, screenshot, game UI, text, logo, watermark, cropped object, duplicate objects, blurry, low quality",            image=init_image,            strength=REDESIGN_STRENGTH,            guidance_scale=REDESIGN_GUIDANCE,            num_inference_steps=REDESIGN_STEPS,            generator=torch.Generator(device="cuda:1").manual_seed(REDESIGN_SEED + index),        ).images[0].convert("RGB")        metrics = image_metrics(original, result)        # If the result remains visually too close, retry once with stronger redesign.        if metrics["ssim_similarity"] >= REDESIGN_RETRY_SIMILARITY:            result = redesign_pipe(                prompt=prompt,                negative_prompt="exact copy, identical design, screenshot, game UI, text, logo, watermark, cropped object, duplicate objects, blurry, low quality",                image=init_image,                strength=0.76,                guidance_scale=7.0,                num_inference_steps=32,                generator=torch.Generator(device="cuda:1").manual_seed(REDESIGN_SEED + 10000 + index),            ).images[0].convert("RGB")            metrics = image_metrics(original, result)        result.save(redesigned_path)        try:            init_image.close()        except Exception:            pass        try:            result.close()        except Exception:            pass        del init_image, result        torch.cuda.empty_cache()        gc.collect()    with Image.open(redesigned_path) as redesigned_file:        redesigned = redesigned_file.convert("RGB")    metrics = image_metrics(original, redesigned)    preview_original = ImageOps.fit(original, (512, 512))    preview_redesigned = ImageOps.fit(redesigned, (512, 512))    preview = Image.new("RGB", (1024, 560), "white")    preview.paste(preview_original, (0, 32))    preview.paste(preview_redesigned, (512, 32))    draw = ImageDraw.Draw(preview)    draw.text((12, 10), "ORIGINAL REFERENCE", fill="black")    draw.text((524, 10), "REDESIGNED IMAGE", fill="black")    preview.save(comparison_path)    report = {        "asset": name,        "source_file": str(original_path),        "original_copy": str(original_copy),        "redesigned_image": str(redesigned_path),        "comparison_preview": str(comparison_path),        "model": REDESIGN_MODEL,        "strength": REDESIGN_STRENGTH,        "metrics": metrics,        "note": "Similarity metrics are visual diagnostics only; they do not determine copyright status or permission to redistribute."    }    redesign_reports.append(report)    item["original_path"] = str(original_copy)    item["redesigned_path"] = str(redesigned_path)    item["name"] = f"{name}_redesigned"  # keep new outputs separate from any older meshes made from the untouched reference    item["path"] = str(redesigned_path)  # existing shape_worker and paint_worker now consume the redesigned image    print(f"{name}: redesigned | SSIM={metrics['ssim_similarity']:.3f} | mean pixel difference={metrics['mean_absolute_pixel_difference']:.1f}")    for image_obj in (original, redesigned, preview_original, preview_redesigned, preview):        try:            image_obj.close()        except Exception:            pass    del original, redesigned, preview_original, preview_redesigned, preview, draw    gc.collect()    torch.cuda.empty_cache()report_path = REDESIGN_ROOT / "redesign_report.json"report_path.write_text(json.dumps(redesign_reports, indent=2), encoding="utf-8")# Release the redesign model before the two Hunyuan3D workers load their own models.del redesign_pipegc.collect()torch.cuda.empty_cache()print(f"Image redesign complete: {len(images)} input(s). Hunyuan3D will now use the redesigned images.")print("Originals, redesigned images, side-by-side previews, and the report are in:", REDESIGN_ROOT)

## 6. Run a resource-aware, resumable batchThe processing order is unchanged. Safeguards monitor host RAM, both GPUs' VRAM and disk space; per-asset temporary buffers are released, and logs remain under `/kaggle/working/hy3d_logs`. Batch size caps session work, while live guards protect peak memory. It never clears prior output or logs. If resources approach the safety floor, workers are stopped and completed files remain available for the next manually started session.

In [ ]:
import json, os, shutil, subprocess, sys, time, gc, structfrom pathlib import Pathimport psutilWORK = "/tmp/hy3d"QUEUE = f"{WORK}/queue"OUT_DIR = Path(OUT_DIR)LOG_DIR = Path("/kaggle/working/hy3d_logs")LOG_DIR.mkdir(parents=True, exist_ok=True)OUT_DIR.mkdir(parents=True, exist_ok=True)# Read actual resources in this Kaggle session; published limits are ceilings, not current free capacity.def gb(n):    return n / (1024 ** 3)vm = psutil.virtual_memory()ram_available_gb = gb(vm.available)work_free_gb = gb(shutil.disk_usage("/kaggle/working").free)tmp_free_gb = gb(shutil.disk_usage("/tmp").free)gpu_lines = subprocess.run(    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader,nounits"],    capture_output=True, text=True, check=True).stdout.strip().splitlines()print(f"Session RAM: total={gb(vm.total):.1f} GB | available={ram_available_gb:.1f} GB")print(f"Disk free: /kaggle/working={work_free_gb:.1f} GB | /tmp={tmp_free_gb:.1f} GB")print("GPU status:")for line in gpu_lines:    print(" ", line)gpu_start_free_gb = []for line in gpu_lines:    try:        gpu_start_free_gb.append(int(line.split(",")[3].strip()) / 1024)    except (IndexError, ValueError):        passif len(gpu_start_free_gb) < 2 or min(gpu_start_free_gb) < 2.0:    raise RuntimeError("Safe stop: both GPUs must be detected with at least 2 GB free VRAM each before workers start.")# Refuse to start if the session is already under pressure. No files are removed.# Conservative operational safety margins; these are not Kaggle's published hard limits.RAM_START_FLOOR_GB, RAM_STOP_FLOOR_GB = 14.0, 8.0DISK_START_FLOOR_GB, DISK_STOP_FLOOR_GB = 3.0, 1.5GPU_STOP_FLOOR_GB = 0.75if ram_available_gb < RAM_START_FLOOR_GB:    raise RuntimeError(f"Safe stop: RAM available {ram_available_gb:.1f} GB is below {RAM_START_FLOOR_GB:.1f} GB.")if work_free_gb < DISK_START_FLOOR_GB or tmp_free_gb < DISK_START_FLOOR_GB:    raise RuntimeError("Safe stop: less than 3 GB free on a required working disk.")# Session batch cap limits total work; live RAM/VRAM guards protect per-item peak usage.if ram_available_gb < 20:    batch_size = 1elif ram_available_gb < 25:    batch_size = 2else:    batch_size = 3print(f"Adaptive session batch cap: {batch_size} asset(s).")# Load the durable upload record from the private Hugging Face Dataset.from kaggle_secrets import UserSecretsClientfrom huggingface_hub import HfApiDATASET_ID = "ZzwahwahzZ/clash-of-legends-assets"hf_token = UserSecretsClient().get_secret("HF_TOKEN")if not hf_token:    raise RuntimeError("Kaggle Secret HF_TOKEN is missing. No assets were changed.")api = HfApi(token=hf_token)api.create_repo(repo_id=DATASET_ID, repo_type="dataset", private=True, exist_ok=True)repo_info = api.repo_info(repo_id=DATASET_ID, repo_type="dataset")if not getattr(repo_info, "private", False):    raise RuntimeError("The destination Dataset is not Private. No assets were uploaded.")remote_files = set(api.list_repo_files(repo_id=DATASET_ID, repo_type="dataset"))remote_asset_names = set()for remote_path in remote_files:    parts = remote_path.split("/")    if len(parts) >= 3 and parts[0] == "generated" and parts[-1] == "textured_mesh.glb":        remote_asset_names.add(parts[-2])def valid_glb(path):    try:        path = Path(path)        if not path.is_file() or path.stat().st_size < 1024:            return False        with path.open("rb") as f:            header = f.read(12)        return (len(header) == 12 and header[:4] == b"glTF"                and struct.unpack("<I", header[4:8])[0] == 2                and struct.unpack("<I", header[8:12])[0] == path.stat().st_size)    except OSError:        return False# The image list at this point contains the redesigned inputs from Stage 5A.# Existing local GLBs are retained for Stage 8; remotely verified assets are never regenerated.pending = []local_ready = []for item in images:    asset_name = item["name"]    local_glb = OUT_DIR / asset_name / "textured_mesh.glb"    if valid_glb(local_glb):        local_ready.append(asset_name)        continue    if asset_name in remote_asset_names:        continue    pending.append(item)print(f"Already uploaded remotely: {len(remote_asset_names)} | valid local GLBs awaiting upload: {len(local_ready)}")print(f"Assets still requiring generation: {len(pending)}")batch_names = [x["name"] for x in pending[:batch_size]]batch_images = pending[:batch_size]print("This session's batch:", batch_names if batch_names else "No generation required; proceed to Results and Upload.")# Rebuild only the ephemeral queue from durable local mesh files. Logs live under /kaggle/working.# Never delete output meshes, redesign inputs, prior logs, or the Hugging Face completion record.os.makedirs(WORK, exist_ok=True)shutil.rmtree(QUEUE, ignore_errors=True)os.makedirs(QUEUE, exist_ok=True)run_id = time.strftime("%Y%m%d-%H%M%S")shape_log = str(LOG_DIR / f"shape_{run_id}.log")paint_log = str(LOG_DIR / f"paint_{run_id}.log")def launch(script, gpu, config_path):    log_file = open(shape_log if script == "shape_worker" else paint_log, "a", buffering=1)    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",               HY3DGEN_MODELS=WEIGHTS, U2NET_HOME=f"{WEIGHTS}/u2net")    if OFFLINE:        env["HF_HUB_OFFLINE"] = "1"    proc = subprocess.Popen(        [sys.executable, f"{WORK}/{script}.py", config_path],        stdout=log_file, stderr=subprocess.STDOUT, env=env    )    return proc, log_fileif batch_images:    config_path = f"{WORK}/config_{run_id}.json"    with open(config_path, "w") as f:        json.dump(dict(SETTINGS, repo=REPO, queue=QUEUE, out=str(OUT_DIR),                       weights=WEIGHTS, images=batch_images), f)    n_gpus = len(gpu_lines)    if n_gpus < 2:        raise RuntimeError("This notebook requires two T4 GPUs; no worker was started.")    shape, shape_handle = launch("shape_worker", 0, config_path)    paint = None    paint_handle = None    started = time.time()    last_stats = 0    stopped_for_pressure = False    def pump(path, offset):        if not os.path.exists(path):            return offset        with open(path, errors="replace") as f:            f.seek(offset)            chunk = f.read()            offset = f.tell()        for line in chunk.replace("\r", "\n").splitlines():            if line.strip() and not any(s in line for s in ("it/s]", "s/it]", "%|")):                print(line)        return offset    offsets = {shape_log: 0, paint_log: 0}    while True:        offsets[shape_log] = pump(shape_log, offsets[shape_log])        if paint is not None:            offsets[paint_log] = pump(paint_log, offsets[paint_log])        if shape.poll() is not None:            # A durable local white mesh is the recovery point; the queue itself is disposable.            open(f"{QUEUE}/_SHAPE_DONE", "a").close()            if shape.returncode != 0 and paint is None:                print("Shape worker exited before the paint worker could start; inspect the preserved log.")                break        if paint is None and os.path.exists(f"{QUEUE}/_SHAPE_READY"):            paint, paint_handle = launch("paint_worker", 1, config_path)        if shape.poll() is not None and paint is not None and paint.poll() is not None:            break        now = time.time()        if now - last_stats >= 10:            last_stats = now            current_ram = gb(psutil.virtual_memory().available)            current_work = gb(shutil.disk_usage("/kaggle/working").free)            current_tmp = gb(shutil.disk_usage("/tmp").free)            gpu_now = subprocess.run(                ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],                capture_output=True, text=True            )            gpu_free_gb = []            if gpu_now.returncode == 0:                for row in gpu_now.stdout.strip().splitlines():                    try:                        gpu_free_gb.append(int(row.split(",")[1].strip()) / 1024)                    except (IndexError, ValueError):                        pass            gpu_report = ", ".join(f"GPU{i} free={v:.2f} GB" for i, v in enumerate(gpu_free_gb)) or "GPU VRAM unavailable"            print(f"--- elapsed={now-started:.0f}s | RAM available={current_ram:.1f} GB | "                  f"disk free working={current_work:.1f} GB / tmp={current_tmp:.1f} GB | {gpu_report}")            gpu_pressure = bool(gpu_free_gb) and min(gpu_free_gb) < GPU_STOP_FLOOR_GB            # Stop before host RAM, disk, or either GPU is exhausted.            if current_ram < RAM_STOP_FLOOR_GB or current_work < DISK_STOP_FLOOR_GB or current_tmp < DISK_STOP_FLOOR_GB or gpu_pressure:                print("RESOURCE GUARD: stopping workers before further memory/disk pressure. Existing output is preserved.")                stopped_for_pressure = True                for proc in (shape, paint):                    if proc is not None and proc.poll() is None:                        proc.terminate()                for proc in (shape, paint):                    if proc is not None and proc.poll() is None:                        try:                            proc.wait(timeout=20)                        except subprocess.TimeoutExpired:                            proc.kill()                break        time.sleep(3)    # Close log handles; never remove these logs.    shape_handle.close()    if paint_handle is not None:        paint_handle.close()    gc.collect()    print(f"Batch ended after {(time.time()-started)/60:.1f} min | shape exit={shape.returncode} | "          f"paint exit={None if paint is None else paint.returncode} | resource_guard={stopped_for_pressure}")else:    print("No new 3D generation batch is needed. Existing local results are kept for upload.")del hf_token, UserSecretsClient

## 7. Inspect and package this session's results
Shows all valid local textured models but packages only the current batch when sufficient disk space is available. Existing output files are not deleted.


In [ ]:
import os, shutil, time, zipfile, struct
from pathlib import Path
from IPython.display import FileLink, display

def valid_result_glb(p):
    try:
        with p.open("rb") as f:
            header = f.read(12)
        return (p.stat().st_size >= 1024 and len(header) == 12 and header[:4] == b"glTF"
                and struct.unpack("<I", header[4:8])[0] == 2
                and struct.unpack("<I", header[8:12])[0] == p.stat().st_size)
    except OSError:
        return False

done = sorted(p for p in Path(OUT_DIR).glob("*/textured_mesh.glb") if valid_result_glb(p))
shape_only = sorted(Path(OUT_DIR).glob("*/white_mesh.glb"))
print("Redesign originals/previews/reports:", "/kaggle/working/hy3d_redesign")
print(f"{len(done)} valid textured model(s) | {len(shape_only)} shape-stage GLB(s)")
for p in done:
    print(f"  {p.stat().st_size / 2**20:6.1f} MB  {p}")

# Package only this session's batch; never make a second copy of the accumulated output tree.
batch_files = [p for name in batch_names for p in (Path(OUT_DIR) / name).glob("*") if p.is_file()]
required_zip_gb = sum(p.stat().st_size for p in batch_files) / (1024 ** 3) * 1.1
free_gb = shutil.disk_usage("/kaggle/working").free / (1024 ** 3)
if batch_files and free_gb > max(2.0, required_zip_gb + 1.0):
    archive = Path("/kaggle/working") / f"hy3d_batch_{time.strftime('%Y%m%d-%H%M%S')}.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as zf:
        for p in batch_files:
            zf.write(p, arcname=str(p.relative_to(Path("/kaggle/working"))))
    print("Current-batch archive:", archive)
    display(FileLink(str(archive)))
else:
    print(f"Batch archive skipped to protect disk space (free={free_gb:.1f} GB). Original outputs remain available.")


### Optional: preview a model in the notebook

In [ ]:
from pathlib import Pathfrom IPython.display import HTML, FileLink, displayif done:    glb = Path(done[0])    size_mb = glb.stat().st_size / (1024 ** 2)    if size_mb <= 128:        import base64        data = base64.b64encode(glb.read_bytes()).decode()        display(HTML(f"""        <script type="module" src="https://unpkg.com/@google/model-viewer@3.5.0/dist/model-viewer.min.js"></script>        <p>{glb}</p>        <model-viewer src="data:model/gltf-binary;base64,{data}" camera-controls auto-rotate                      style="width:100%;height:520px;background:#222"></model-viewer>"""))    else:        print(f"Preview skipped to protect notebook RAM ({size_mb:.1f} MB GLB). Original model file remains intact:")        display(FileLink(str(glb)))

## Troubleshooting
- **`OUT OF GPU MEMORY` in the paint log**: keep `texture_resolution=512` and lower `texture_views` to 6. Shape-stage OOM: lower `octree_resolution` or `num_chunks`.
- **`nvcc not found`**: the session has no GPU attached. Set the accelerator to GPU T4 x2 and restart.
- **Install fails on a package**: Kaggle updates its image regularly. The error usually names the package; loosen that pin in `install.sh` and rerun the cell.
- **Session restarted**: `/tmp` is wiped, so rerun everything. Outputs in `/kaggle/working` survive while the session is alive, and `skip_existing=True` resumes where it stopped.
- Full worker logs: `/tmp/hy3d/shape_worker.log` and `/tmp/hy3d/paint_worker.log`.

## 8. Upload and verify assets in the private project Dataset
The uploader skips assets already present remotely, verifies each exact remote path after upload, and persists a completion checkpoint in `pipeline_state/completed_assets.json`. A session interrupted during upload can safely continue without duplicating verified assets.

Before running this cell, add a Kaggle Secret named `HF_TOKEN` containing your Hugging Face write token. Never paste the token into notebook code or output.


In [ ]:
upload_error = None
try:
    from pathlib import Path
    import json, os, time, struct
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    
    DATASET_ID = "ZzwahwahzZ/clash-of-legends-assets"
    OUT_DIR = Path("/kaggle/working/hy3d_out")
    STATE_PATH = "pipeline_state/completed_assets.json"
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("Kaggle Secret HF_TOKEN is missing. Add it in Add-ons → Secrets; never paste it into a cell.")
    
    api = HfApi(token=hf_token)
    api.create_repo(repo_id=DATASET_ID, repo_type="dataset", private=True, exist_ok=True)
    dataset_info = api.repo_info(repo_id=DATASET_ID, repo_type="dataset")
    if not getattr(dataset_info, "private", False):
        raise RuntimeError("The Hugging Face Dataset is PUBLIC. No upload performed.")
    
    remote_files = set(api.list_repo_files(repo_id=DATASET_ID, repo_type="dataset"))
    uploaded_names = set()
    for remote_path in remote_files:
        parts = remote_path.split("/")
        if len(parts) >= 3 and parts[0] == "generated" and parts[-1] == "textured_mesh.glb":
            uploaded_names.add(parts[-2])
    
    state = {"dataset": DATASET_ID, "updated_utc": None, "completed_assets": {}}
    state_file = OUT_DIR / "completed_assets_state.json"
    if state_file.is_file():
        try:
            state = json.loads(state_file.read_text(encoding="utf-8"))
        except Exception:
            print("Local state file is unreadable; remote repository listing remains the source of truth.")
    state.setdefault("completed_assets", {})
    
    glbs = sorted(OUT_DIR.glob("*/textured_mesh.glb"))
    validated = []
    for p in glbs:
        if p.stat().st_size < 1024:
            print(f"SKIP tiny/invalid file: {p}")
            continue
        with p.open("rb") as f:
            header = f.read(12)
        if (len(header) != 12 or header[:4] != b"glTF"
                or struct.unpack("<I", header[4:8])[0] != 2
                or struct.unpack("<I", header[8:12])[0] != p.stat().st_size):
            print(f"SKIP incomplete/invalid GLB header or declared length: {p}")
            continue
        validated.append(p)
    
    if not validated:
        print("No validated local GLB files are waiting for upload.")
    else:
        for p in validated:
            asset_name = p.parent.name
            dest = f"generated/assets/{asset_name}/textured_mesh.glb"
            if asset_name in uploaded_names:
                # A matching remote asset path is already present; do not upload a duplicate.
                state["completed_assets"][asset_name] = {"dataset_path": next(
                    x for x in remote_files if x.startswith("generated/") and
                    x.endswith(f"/{asset_name}/textured_mesh.glb")), "bytes": p.stat().st_size}
                print(f"ALREADY VERIFIED remotely: {asset_name}")
                continue
    
            api.upload_file(path_or_fileobj=str(p), path_in_repo=dest, repo_id=DATASET_ID,
                            repo_type="dataset", commit_message=f"Add verified Hunyuan3D asset {asset_name}")
            # Verify the exact destination exists after the upload before marking the asset complete.
            refreshed = set(api.list_repo_files(repo_id=DATASET_ID, repo_type="dataset"))
            if dest not in refreshed:
                raise RuntimeError(f"Upload could not be verified for {asset_name}; local output is preserved.")
            remote_files = refreshed
            uploaded_names.add(asset_name)
            state["completed_assets"][asset_name] = {"dataset_path": dest, "bytes": p.stat().st_size}
            print(f"VERIFIED: {asset_name} -> {dest}")
    
            # Persist progress after each individually verified asset, so an interrupted session resumes safely.
            state["updated_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
            state_file.write_text(json.dumps(state, indent=2), encoding="utf-8")
            api.upload_file(path_or_fileobj=str(state_file), path_in_repo=STATE_PATH,
                            repo_id=DATASET_ID, repo_type="dataset",
                            commit_message="Update verified Hunyuan3D pipeline checkpoint")
    
        state["updated_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        state_file.write_text(json.dumps(state, indent=2), encoding="utf-8")
        api.upload_file(path_or_fileobj=str(state_file), path_in_repo=STATE_PATH,
                        repo_id=DATASET_ID, repo_type="dataset",
                        commit_message="Save Hunyuan3D pipeline checkpoint")
        print(f"Checkpoint saved. Verified remote assets: {len(state['completed_assets'])}")
        print(f"Private Dataset: https://huggingface.co/datasets/{DATASET_ID}")
    
    del hf_token, secrets
    
except Exception as e:
    upload_error = f"{type(e).__name__}: {e}"
    print("Hugging Face upload/checkpoint step failed; local generated files are preserved.")
    print(upload_error)


In [ ]:
# FINAL SESSION REPORT — uploads a small JSON summary to this GitHub repository.
# Requires Kaggle Secret GITHUB_TOKEN with Contents: Read and write for this repository.
# This cell does not alter, move, or delete any generated 3D assets.
import os, json, time, shutil, subprocess, traceback
from pathlib import Path
import requests

REPORT_REPO = "ZzwahwahzZ/Clash-of-Legends"
REPORT_BRANCH = "main"
REPORT_DIR = "tools/ai-asset-generation/reports"
OUT = Path("/kaggle/working/hy3d_out")
LOGS = Path("/kaggle/working/hy3d_logs")
REPORT_DIR_LOCAL = Path("/kaggle/working/pipeline_reports")
REPORT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

def _valid_glb(p):
    try:
        import struct
        with p.open("rb") as f: h = f.read(12)
        return p.stat().st_size >= 1024 and len(h) == 12 and h[:4] == b"glTF" and struct.unpack("<I", h[4:8])[0] == 2 and struct.unpack("<I", h[8:12])[0] == p.stat().st_size
    except Exception:
        return False

def _tail(path, limit=80):
    try: return path.read_text(errors="replace").splitlines()[-limit:]
    except Exception as e: return [f"Could not read log: {type(e).__name__}: {e}"]

def _gpu_status():
    try:
        p = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv"], capture_output=True, text=True, timeout=15)
        return (p.stdout or p.stderr).strip().splitlines()
    except Exception as e: return [f"unavailable: {type(e).__name__}: {e}"]

try:
    import psutil
    vm = psutil.virtual_memory()
    ram = {"total_gb": round(vm.total / 2**30, 2), "available_gb": round(vm.available / 2**30, 2)}
except Exception as e:
    ram = {"error": f"{type(e).__name__}: {e}"}

try:
    disk = {p: {"free_gb": round(shutil.disk_usage(p).free / 2**30, 2), "total_gb": round(shutil.disk_usage(p).total / 2**30, 2)} for p in ["/kaggle/working", "/tmp"]}
except Exception as e:
    disk = {"error": f"{type(e).__name__}: {e}"}

textured = sorted(p for p in OUT.glob("*/textured_mesh.glb") if _valid_glb(p)) if OUT.exists() else []
white = sorted(OUT.glob("*/white_mesh.glb")) if OUT.exists() else []
logs = sorted(LOGS.glob("*.log"), key=lambda p: p.stat().st_mtime) if LOGS.exists() else []
state_path = OUT / "completed_assets_state.json"
try:
    upload_state = json.loads(state_path.read_text(encoding="utf-8")) if state_path.is_file() else {}
except Exception as e:
    upload_state = {"read_error": f"{type(e).__name__}: {e}"}

report = {
    "schema_version": 1,
    "project": "Clash of Legends",
    "repository": REPORT_REPO,
    "reported_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "session": {
        "kaggle_working_exists": Path("/kaggle/working").exists(),
        "run_id": globals().get("run_id"),
        "batch_names": globals().get("batch_names", []),
        "resource_guard_stopped_workers": globals().get("stopped_for_pressure"),
        "shape_exit_code": getattr(globals().get("shape"), "returncode", None),
        "paint_exit_code": getattr(globals().get("paint"), "returncode", None),
        "duration_seconds": round(time.time() - globals().get("started", time.time()), 1) if "started" in globals() else None,
    },
    "outputs": {
        "validated_textured_glb_count": len(textured),
        "textured_glb": [{"asset": p.parent.name, "path": str(p), "bytes": p.stat().st_size} for p in textured],
        "white_mesh_glb_count": len(white),
        "white_mesh_names": [p.parent.name for p in white],
    },
    "huggingface_upload_checkpoint": upload_state,
    "huggingface_upload_error": globals().get("upload_error"),
    "resources_at_report_time": {"ram": ram, "disk": disk, "gpu": _gpu_status()},
    "worker_logs": [{"name": p.name, "tail": _tail(p)} for p in logs[-6:]],
}
try:
    ip = get_ipython()
    last_error = getattr(getattr(ip, "last_execution_result", None), "error_in_exec", None)
    if last_error is not None:
        report["last_cell_error"] = {"type": type(last_error).__name__, "message": str(last_error)}
except Exception:
    pass

stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
local_report = REPORT_DIR_LOCAL / f"kaggle-session-{stamp}.json"
local_report.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
(REPORT_DIR_LOCAL / "latest.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Local session report saved:", local_report)

# The credential is read only from Kaggle Secrets and is never printed or included in the report.
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    github_token = None

if not github_token:
    print("GitHub report not uploaded: add Kaggle Secret GITHUB_TOKEN (repo Contents read/write), then rerun this report cell.")
else:
    headers = {"Authorization": f"Bearer {github_token}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
    def _put_report(path, content, message):
        url = f"https://api.github.com/repos/{REPORT_REPO}/contents/{path}"
        old = requests.get(url, headers=headers, params={"ref": REPORT_BRANCH}, timeout=30)
        payload = {"message": message, "content": __import__("base64").b64encode(content.encode("utf-8")).decode("ascii"), "branch": REPORT_BRANCH}
        if old.status_code == 200:
            payload["sha"] = old.json()["sha"]
        elif old.status_code != 404:
            raise RuntimeError(f"GitHub read failed ({old.status_code}): {old.text[:500]}")
        put = requests.put(url, headers=headers, json=payload, timeout=60)
        if put.status_code not in (200, 201):
            raise RuntimeError(f"GitHub report upload failed ({put.status_code}): {put.text[:700]}")
        return put.json().get("content", {}).get("html_url")

    try:
        latest_url = _put_report(f"{REPORT_DIR}/kaggle-session-latest.json", json.dumps(report, ensure_ascii=False, indent=2), "Update latest Kaggle session report")
        archive_url = _put_report(f"{REPORT_DIR}/kaggle-session-{stamp}.json", json.dumps(report, ensure_ascii=False, indent=2), "Add Kaggle session report")
        print("GitHub report uploaded successfully.")
        print("Latest report:", latest_url)
        print("Archived report:", archive_url)
    except Exception as e:
        print(f"GitHub report upload failed; local report is preserved: {type(e).__name__}: {e}")
    finally:
        github_token = None
